In [ ]:
import sys
import json
import os
from pathlib import Path

# Project layout on local WSL
PROJECT_ROOT = "/mnt/d/Onedrive/ICPBR/Alberta/koopman_events/LFP_BOLD_kpm_polished/python_scripts/autodl_local_e10gb1"
SOLVER_DIR = PROJECT_ROOT
DATA_ROOT = "/mnt/e/DataPons_processed"
RESULTS_ROOT = "/mnt/e/autodl_results/e10gb1/mlp"
OUTPUT_PARENT = os.path.join(RESULTS_ROOT, "outputs")
CKPT_PARENT = os.path.join(RESULTS_ROOT, "checkpoints")
LOG_PARENT = os.path.join(RESULTS_ROOT, "logs")
RUN_NAME_BASE = "mlp_obs"
EXPERIMENT_NAME = "e10gb1_260415_shuffle_plateau"

for path in [PROJECT_ROOT, SOLVER_DIR]:
    if path not in sys.path:
        sys.path.append(path)

for path in [DATA_ROOT, OUTPUT_PARENT, CKPT_PARENT, LOG_PARENT]:
    os.makedirs(path, exist_ok=True)

# Backward-compatible placeholders. These are finalized after observable_mode is set.
project_root = PROJECT_ROOT
solver_dir = SOLVER_DIR
data_root = DATA_ROOT
output_root = None
checkpoint_root = None
log_root = None
checkpoint_path = None
best_checkpoint_path = None

print(f"project_root: {project_root}")
print(f"solver_dir: {solver_dir}")
print(f"data_root: {data_root}")
print(f"output_parent: {OUTPUT_PARENT}")
print(f"checkpoint_parent: {CKPT_PARENT}")
print(f"log_parent: {LOG_PARENT}")
print(f"run_name_base: {RUN_NAME_BASE}")
print(f"experiment_name: {EXPERIMENT_NAME}")


In [ ]:
import tensorflow as tf
import edmd_utils

# Use GPU on AutoDL
selected_device = 'gpu'

edmd_utils.set_device(selected_device)


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from scipy.linalg import svd

In [ ]:
# koopmanlib is already installed by pip in python directories
from koopmanlib.dictionary import PsiNN

solver_name = 'resdmd_batch'
residual_form = 'projected_kv'  # switch to 'projected_vlambda' to compare the alternative residual
print(f"residual_form: {residual_form}")
KoopmanSolver = edmd_utils.import_solver(solver_name)

In [ ]:
basis_function = PsiNN(layer_sizes=[100,100,100], n_psi_train=100)

In [ ]:
# Data configuration
# Only `obs` is used from the MAT file.
data_subdir = "e10gb1/reskoopnet_dictionary"
observable_mode = "abs"  # supported: "abs", "complex", "complex_split"

observable_mode_map = {
    "abs": "abs",
    "complex": "complex_split",
    "complex_split": "complex_split",
}
observable_tag = observable_mode_map[observable_mode]
data_filename = f"e10gb1_low50_high250_g2_{observable_tag}_single.mat"

# Build experiment-specific writable paths so different runs stay separate.
residual_form_tag = str(residual_form).replace("/", "-").replace(" ", "_")
effective_experiment_name = f"{EXPERIMENT_NAME}_{residual_form_tag}"
run_label = f"{RUN_NAME_BASE}_{effective_experiment_name}_{observable_tag}"
OUTPUT_ROOT = os.path.join(OUTPUT_PARENT, run_label)
CKPT_ROOT = os.path.join(CKPT_PARENT, run_label)
LOG_ROOT = os.path.join(LOG_PARENT, run_label)

for path in [
    OUTPUT_ROOT,
    CKPT_ROOT,
    LOG_ROOT,
    os.path.join(CKPT_ROOT, "final"),
    os.path.join(CKPT_ROOT, "best")
]:
    os.makedirs(path, exist_ok=True)

# Backward-compatible aliases used by the rest of the notebook.
output_root = OUTPUT_ROOT
checkpoint_root = CKPT_ROOT
log_root = LOG_ROOT
checkpoint_path = os.path.join(CKPT_ROOT, "final", "model.ckpt")
best_checkpoint_path = os.path.join(CKPT_ROOT, "best", "model.ckpt")
final_training_state_path = os.path.join(CKPT_ROOT, "final", "training_state.json")
best_training_state_path = os.path.join(CKPT_ROOT, "best", "training_state.json")

# Large MATLAB files are usually saved as v7.3 / HDF5 even if the suffix is .mat.
file_type = ".h5"
field_name = "obs"

data_input_path = os.path.join(data_root, data_subdir)
data_full_path = os.path.join(data_input_path, data_filename)

print(f"observable_mode: {observable_mode} -> file tag: {observable_tag}")
print(f"experiment_name: {EXPERIMENT_NAME}")
print(f"residual_form: {residual_form}")
print(f"effective_experiment_name: {effective_experiment_name}")
print(f"run_label: {run_label}")
print(f"output_root: {output_root}")
print(f"checkpoint_root: {checkpoint_root}")
print(f"log_root: {log_root}")
print(f"checkpoint_path: {checkpoint_path}")
print(f"best_checkpoint_path: {best_checkpoint_path}")
print(f"final_training_state_path: {final_training_state_path}")
print(f"best_training_state_path: {best_training_state_path}")
print(f"data_full_path: {data_full_path}")


In [ ]:
# Optional: inspect top-level keys once.
# For your current files, the main numeric array of interest is expected to be `obs`.
import h5py

with h5py.File(data_full_path, "r") as f:
    print(list(f.keys()))

In [ ]:
# Load only the numeric observable matrix `obs`.
# Keep the array in float32 to reduce memory pressure for large observables.
# The transpose keeps the downstream pipeline in the expected [time, features] layout.
DATA = edmd_utils.load_data(
    data_filename,
    data_input_path,
    file_type,
    field_name
).T.astype(np.float32, copy=False)

print("Loaded DATA shape:", DATA.shape)
print("Loaded DATA dtype:", DATA.dtype)
print(f"Loaded DATA size: {DATA.nbytes / (1024 ** 3):.2f} GiB")

In [ ]:
reduce_flag = False
if reduce_flag:
    n_modes = 10
    DATA_reduced = edmd_utils.dim_reduction(DATA, n_modes)

In [ ]:
import gc

data_train, data_valid = edmd_utils.transfer_data_format(DATA, train_ratio=0.7)
print("Data shape: ", data_train[1].shape)

# Release the full observable matrix once the train/valid split is materialized.
del DATA
gc.collect()

In [ ]:
solver = KoopmanSolver(dic=basis_function,
                         target_dim=data_train[0].shape[-1],
                         reg=0.1,
                         residual_form=residual_form)

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import scipy.io
import tensorflow as tf

N_round = 1
# extra_epochs means additional outer epochs for this run.
# For a fresh experiment with no existing checkpoint, it is the total outer epoch count to run.
extra_epochs = 40
train_batch_size = 2000
initial_lr = 1e-4
inner_epochs = 3
outer_lr_decay_factor = 0.5
outer_lr_patience = 8
outer_lr_cooldown = 2
outer_min_lr = 1e-5
train_shuffle = True
shuffle_buffer_size = 200000
shuffle_seed = 42
reshuffle_each_iteration = True
resume_mode = "fresh"  # supported: "fresh", "final", "best"
clear_existing_checkpoints = False
completed_rounds = int(globals().get("completed_rounds", 0))
current_history_context = {
    "effective_experiment_name": effective_experiment_name,
    "observable_mode": observable_mode,
    "residual_form": residual_form,
    "run_label": run_label,
}
previous_history_context = globals().get("_training_history_context", None)
same_history_context = previous_history_context == current_history_context

def _load_previous_loss_history(summary_root, observable_filename):
    out_base = Path(observable_filename).stem
    summary_candidates = sorted(
        Path(summary_root).glob(f"{out_base}_Python_resdmd_Layer_100_Ndict_*_summary.mat"),
        key=lambda path: path.stat().st_mtime
    )
    if not summary_candidates:
        return [], [], None

    latest_summary = summary_candidates[-1]
    try:
        mat = scipy.io.loadmat(latest_summary, squeeze_me=True, struct_as_record=False)
        edmd_outputs = mat.get("EDMD_outputs", None)
        if edmd_outputs is None:
            return [], [], latest_summary

        prev_loss = np.atleast_1d(np.asarray(getattr(edmd_outputs, "loss", []), dtype=np.float32)).ravel().tolist()
        prev_val_loss = np.atleast_1d(np.asarray(getattr(edmd_outputs, "val_loss", []), dtype=np.float32)).ravel().tolist()
        return prev_loss, prev_val_loss, latest_summary
    except Exception as exc:
        print(f"Could not load previous loss history from {latest_summary}: {exc}")
        return [], [], latest_summary

def _load_training_state(state_path):
    if state_path is None or (not os.path.exists(state_path)):
        return None
    try:
        with open(state_path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        print(f"Could not load trainer state from {state_path}: {exc}")
        return None

def _as_scalar_or_nan(value):
    try:
        if value is None:
            return np.nan
        arr = np.asarray(value).squeeze()
        if arr.size == 0:
            return np.nan
        if np.iscomplexobj(arr):
            arr = np.real(arr)
        return float(arr.item())
    except Exception:
        return np.nan

def _history_array(history, key):
    return np.asarray(
        [_as_scalar_or_nan(item.get(key, np.nan)) for item in history],
        dtype=np.float32
    )

def _infer_summary_ndict(current_solver):
    eigvals = getattr(current_solver, "eigenvalues", None)
    if eigvals is not None:
        eigvals_arr = np.asarray(eigvals).ravel()
        if eigvals_arr.size > 0:
            return int(eigvals_arr.size)

    basis_func = getattr(current_solver, "basis_func", None)
    n_psi_train = getattr(basis_func, "n_psi_train", None)
    target_dim = getattr(current_solver, "target_dim", None)
    if n_psi_train is not None and target_dim is not None:
        return int(n_psi_train + target_dim + 1)
    if n_psi_train is not None:
        return int(n_psi_train)
    return -1

def _build_training_summary_outputs(current_solver, train_loss_history, val_loss_history):
    outer_history = getattr(current_solver, "outer_history", None) or []
    residual_form_value = str(getattr(current_solver, "residual_form", ""))
    final_eigvec_cond = np.float32(_as_scalar_or_nan(getattr(current_solver, "eigvec_cond", np.nan)))
    best_checkpoint_value = str(getattr(current_solver, "best_checkpoint_path", "") or "")
    final_checkpoint_value = str(getattr(current_solver, "final_checkpoint_path", "") or "")
    num_outer_epochs = int(len(outer_history))

    if num_outer_epochs > 0:
        outer_epoch_history = _history_array(outer_history, "outer_epoch")
        outer_train_metric_history = _history_array(outer_history, "train_metric")
        outer_val_metric_history = _history_array(outer_history, "val_metric")
        outer_eigvec_cond_history = _history_array(outer_history, "eigvec_cond")
        outer_lr_history = _history_array(outer_history, "lr")
        outer_reg_history = _history_array(outer_history, "reg")
        outer_inner_train_last_history = _history_array(outer_history, "inner_train_last")
        outer_inner_val_last_history = _history_array(outer_history, "inner_val_last")

        finite_val_mask = np.isfinite(outer_val_metric_history)
        if np.any(finite_val_mask):
            val_candidates = np.where(finite_val_mask, outer_val_metric_history, np.inf)
            best_idx = int(np.argmin(val_candidates))
            best_outer_index = np.float32(best_idx)
            best_outer_epoch = np.float32(outer_epoch_history[best_idx])
            best_val_metric = np.float32(outer_val_metric_history[best_idx])
            best_train_metric_at_best = np.float32(outer_train_metric_history[best_idx])
            best_eigvec_cond_at_best = np.float32(outer_eigvec_cond_history[best_idx])
            best_lr_at_best = np.float32(outer_lr_history[best_idx])
            best_reg_at_best = np.float32(outer_reg_history[best_idx])
        else:
            best_outer_index = np.nan
            best_outer_epoch = np.nan
            best_val_metric = np.nan
            best_train_metric_at_best = np.nan
            best_eigvec_cond_at_best = np.nan
            best_lr_at_best = np.nan
            best_reg_at_best = np.nan
    else:
        empty_history = np.array([], dtype=np.float32)
        outer_epoch_history = empty_history
        outer_train_metric_history = empty_history
        outer_val_metric_history = empty_history
        outer_eigvec_cond_history = empty_history
        outer_lr_history = empty_history
        outer_reg_history = empty_history
        outer_inner_train_last_history = empty_history
        outer_inner_val_last_history = empty_history
        best_outer_index = np.nan
        best_outer_epoch = np.nan
        best_val_metric = np.nan
        best_train_metric_at_best = np.nan
        best_eigvec_cond_at_best = np.nan
        best_lr_at_best = np.nan
        best_reg_at_best = np.nan

    shared_outputs = {
        "loss": np.asarray(train_loss_history, dtype=np.float32),
        "val_loss": np.asarray(val_loss_history, dtype=np.float32),
        "residual_form": residual_form_value,
        "num_outer_epochs": num_outer_epochs,
        "outer_epoch_history": outer_epoch_history,
        "outer_train_metric_history": outer_train_metric_history,
        "outer_val_metric_history": outer_val_metric_history,
        "outer_eigvec_cond_history": outer_eigvec_cond_history,
        "outer_lr_history": outer_lr_history,
        "outer_reg_history": outer_reg_history,
        "outer_inner_train_last_history": outer_inner_train_last_history,
        "outer_inner_val_last_history": outer_inner_val_last_history,
        "best_outer_index": best_outer_index,
        "best_outer_epoch": best_outer_epoch,
        "best_val_metric": best_val_metric,
        "best_train_metric_at_best": best_train_metric_at_best,
        "best_eigvec_cond_at_best": best_eigvec_cond_at_best,
        "best_lr_at_best": best_lr_at_best,
        "best_reg_at_best": best_reg_at_best,
        "final_eigvec_cond": final_eigvec_cond,
        "best_checkpoint_path": best_checkpoint_value,
        "final_checkpoint_path": final_checkpoint_value,
    }

    eigvals = getattr(current_solver, "eigenvalues", None)
    if eigvals is not None:
        shared_outputs["evalues"] = np.asarray(eigvals, dtype=np.complex64)

    modes = getattr(current_solver, "modes", None)
    if modes is not None:
        shared_outputs["kpm_modes"] = np.asarray(modes.T, dtype=np.complex64)

    return shared_outputs

def _save_training_summary(current_solver, train_loss_history, val_loss_history, summary_root, observable_filename):
    try:
        os.makedirs(summary_root, exist_ok=True)
        out_base = Path(observable_filename).stem
        n_dict_value = _infer_summary_ndict(current_solver)
        shared_outputs = _build_training_summary_outputs(current_solver, train_loss_history, val_loss_history)
        shared_outputs["N_dict"] = np.int32(n_dict_value)
        summary_path = os.path.join(
            summary_root,
            f"{out_base}_Python_resdmd_Layer_100_Ndict_{n_dict_value}_summary.mat"
        )
        scipy.io.savemat(summary_path, {"EDMD_outputs": shared_outputs})
        return summary_path
    except Exception as exc:
        print(f"Could not autosave summary: {exc}")
        return None

checkpoint_cleanup_root = CKPT_ROOT
resume_mode = str(resume_mode).strip().lower()
if resume_mode not in {"fresh", "final", "best"}:
    raise ValueError("resume_mode must be one of: 'fresh', 'final', 'best'.")

requested_resume_mode = resume_mode
selected_training_state_path = None
selected_checkpoint_prefix = None
loaded_training_state = None

print(f"Active experiment: {EXPERIMENT_NAME}")
print(f"Experiment checkpoint root: {checkpoint_cleanup_root}")
if clear_existing_checkpoints:
    edmd_utils.remove_checkpoint(checkpoint_cleanup_root)
    loss_all = []
    val_loss_all = []
    completed_rounds = 0
    requested_resume_mode = "fresh"
    print("Checkpoint cleanup requested. Forcing resume_mode='fresh'.")
else:
    if requested_resume_mode == "fresh":
        loss_all = []
        val_loss_all = []
        completed_rounds = 0
        print("Fresh run requested. Clearing in-memory training history for this notebook session.")
    elif (not same_history_context) and (("loss_all" in globals() and len(loss_all) > 0) or ("val_loss_all" in globals() and len(val_loss_all) > 0)):
        loss_all = []
        val_loss_all = []
        completed_rounds = 0
        print("History context changed. Dropping in-memory loss history to avoid mixing experiments.")
    else:
        loss_all = []
        val_loss_all = []

os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
os.makedirs(os.path.dirname(best_checkpoint_path), exist_ok=True)
os.makedirs(LOG_ROOT, exist_ok=True)

latest_final_checkpoint = tf.train.latest_checkpoint(os.path.dirname(checkpoint_path))
latest_best_checkpoint = tf.train.latest_checkpoint(os.path.dirname(best_checkpoint_path))

if requested_resume_mode == "final":
    selected_checkpoint_prefix = checkpoint_path
    selected_training_state_path = final_training_state_path
elif requested_resume_mode == "best":
    selected_checkpoint_prefix = best_checkpoint_path
    selected_training_state_path = best_training_state_path

resume_training = False
effective_resume_mode = "fresh"
selected_latest_checkpoint = None
if requested_resume_mode == "fresh":
    print("Resume mode is 'fresh'; no checkpoint will be restored before training.")
else:
    selected_latest_checkpoint = tf.train.latest_checkpoint(os.path.dirname(selected_checkpoint_prefix))
    if selected_latest_checkpoint is not None:
        resume_training = True
        effective_resume_mode = requested_resume_mode
        loaded_training_state = _load_training_state(selected_training_state_path)
    else:
        print(f"No {requested_resume_mode} checkpoint found. Falling back to fresh training.")

if requested_resume_mode == "fresh":
    loss_all = []
    val_loss_all = []
    completed_rounds = 0
elif resume_training:
    if loaded_training_state is not None:
        loss_all = [float(x) for x in loaded_training_state.get("loss_history", [])]
        val_loss_all = [float(x) for x in loaded_training_state.get("val_loss_history", [])]
        completed_rounds = int(loaded_training_state.get("completed_rounds", 0))
        print(f"Loaded training history from trainer state: {selected_training_state_path}")
    elif effective_resume_mode == "best":
        loss_all = []
        val_loss_all = []
        completed_rounds = 0
        print("Best checkpoint resume has no matching trainer state yet; starting loss history fresh to avoid mixing later final-run history.")
    else:
        loss_all, val_loss_all, latest_summary_file = _load_previous_loss_history(output_root, data_filename)
        completed_rounds = 0
        if latest_summary_file is not None:
            print(f"Loaded previous loss history from: {latest_summary_file}")
        else:
            print("No previous loss summary found. Starting with empty history.")
else:
    loss_all = []
    val_loss_all = []
    completed_rounds = 0
    if requested_resume_mode != "fresh":
        print(f"Starting fresh because no {requested_resume_mode} checkpoint was found.")

globals()["_training_history_context"] = current_history_context
print(f"requested_resume_mode: {resume_mode}")
print(f"same_history_context: {same_history_context}")
print(f"effective_resume_mode: {effective_resume_mode}")
print(f"resume_training: {resume_training}")
if latest_final_checkpoint is not None:
    print(f"Latest final checkpoint: {latest_final_checkpoint}")
if latest_best_checkpoint is not None:
    print(f"Latest best checkpoint: {latest_best_checkpoint}")
if selected_training_state_path is not None:
    print(f"selected_training_state_path: {selected_training_state_path}")
print(f"Current history lengths: train={len(loss_all)}, val={len(val_loss_all)}")
print(f"train_batch_size: {train_batch_size}")
print(f"initial_lr: {initial_lr}")
print(f"inner_epochs: {inner_epochs}")
print(f"train_shuffle: {train_shuffle}, shuffle_buffer_size: {shuffle_buffer_size}, shuffle_seed: {shuffle_seed}, reshuffle_each_iteration: {reshuffle_each_iteration}")
print(f"outer_lr_decay_factor: {outer_lr_decay_factor}, outer_lr_patience: {outer_lr_patience}, outer_lr_cooldown: {outer_lr_cooldown}, outer_min_lr: {outer_min_lr}")


In [ ]:
KoopmanSolver = edmd_utils.import_solver(solver_name)

In [ ]:
stop_flag = False

starting_completed_rounds = int(completed_rounds)

for n_round in range(N_round):
    round_number = starting_completed_rounds + n_round
    print('Round number: ', round_number)
    current_run_metadata = {
        'experiment_name': EXPERIMENT_NAME,
        'effective_experiment_name': effective_experiment_name,
        'run_label': run_label,
        'observable_mode': observable_mode,
        'observable_tag': observable_tag,
        'residual_form': residual_form,
        'train_batch_size': train_batch_size,
        'initial_lr': initial_lr,
        'inner_epochs': inner_epochs,
        'train_shuffle': train_shuffle,
        'shuffle_buffer_size': shuffle_buffer_size,
        'shuffle_seed': shuffle_seed,
        'reshuffle_each_iteration': reshuffle_each_iteration,
        'outer_lr_decay_factor': outer_lr_decay_factor,
        'outer_lr_patience': outer_lr_patience,
        'outer_lr_cooldown': outer_lr_cooldown,
        'outer_min_lr': outer_min_lr,
        'requested_resume_mode': resume_mode,
        'effective_resume_mode': effective_resume_mode,
        'round_number': round_number,
        'completed_rounds': round_number + 1,
    }

    temp_loss, temp_val_loss, stop_flag, lr_changes = solver.build(
        data_train=data_train,
        data_valid=data_valid,
        epochs=extra_epochs,
        batch_size=train_batch_size,
        lr=initial_lr,
        log_interval=1,
        lr_decay_factor=outer_lr_decay_factor,
        Nepoch=inner_epochs,
        end_condition=1e-9,
        train_shuffle=train_shuffle,
        shuffle_buffer_size=shuffle_buffer_size,
        shuffle_seed=shuffle_seed,
        reshuffle_each_iteration=reshuffle_each_iteration,
        outer_lr_patience=outer_lr_patience,
        outer_lr_cooldown=outer_lr_cooldown,
        outer_min_lr=outer_min_lr,
        checkpoint_path=checkpoint_path,
        best_checkpoint_path=best_checkpoint_path,
        resume=resume_training,
        resume_mode=effective_resume_mode,
        final_state_path=final_training_state_path,
        best_state_path=best_training_state_path,
        run_metadata=current_run_metadata)

    loss_all = list(temp_loss)
    val_loss_all = list(temp_val_loss)
    completed_rounds = round_number + 1
    globals()["_training_history_context"] = current_history_context
    resume_training = True
    effective_resume_mode = 'final'
    autosaved_summary = _save_training_summary(
        solver,
        loss_all,
        val_loss_all,
        output_root,
        data_filename,
    )
    if autosaved_summary is not None:
        print(f"autosaved summary: {autosaved_summary}")
    if stop_flag:
        break


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

loss_arr = np.asarray(loss_all, dtype=float)
val_loss_arr = np.asarray(val_loss_all, dtype=float)
outer_history = getattr(solver, "outer_history", None) or []

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

if loss_arr.size > 0:
    axes[0].plot(np.arange(loss_arr.size), loss_arr, linewidth=1.5, color='tab:blue')
    axes[0].set_title("Inner Train Loss")
    axes[0].set_xlabel("inner step")
    axes[0].set_ylabel("loss")
else:
    axes[0].text(0.5, 0.5, "No train loss data", ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title("Inner Train Loss")

if val_loss_arr.size > 0:
    axes[1].plot(np.arange(val_loss_arr.size), val_loss_arr, linewidth=1.5, color='tab:orange')
    axes[1].set_title("Inner Validation Loss")
    axes[1].set_xlabel("inner step")
    axes[1].set_ylabel("val_loss")
else:
    axes[1].text(0.5, 0.5, "No validation loss data", ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title("Inner Validation Loss")

if outer_history:
    outer_epoch = np.asarray([row.get("outer_epoch", np.nan) for row in outer_history], dtype=float)
    train_metric = np.asarray([row.get("train_metric", np.nan) for row in outer_history], dtype=float)
    val_metric = np.asarray([row.get("val_metric", np.nan) for row in outer_history], dtype=float)
    eigvec_cond = np.asarray([row.get("eigvec_cond", np.nan) for row in outer_history], dtype=float)

    axes[2].plot(outer_epoch, train_metric, marker='o', markersize=3, linewidth=1.5, label="train_metric")
    axes[2].plot(outer_epoch, val_metric, marker='o', markersize=3, linewidth=1.5, label="val_metric")
    axes[2].set_title("Outer Spectral Metrics")
    axes[2].set_xlabel("outer_epoch")
    axes[2].set_ylabel("metric")
    axes[2].legend()

    axes[3].plot(outer_epoch, eigvec_cond, marker='o', markersize=3, linewidth=1.5, color='tab:green', label="eigvec_cond")
    if np.all(np.isfinite(eigvec_cond)) and np.all(eigvec_cond > 0):
        axes[3].set_yscale('log')
    axes[3].set_title("Eigenvector Condition Number")
    axes[3].set_xlabel("outer_epoch")
    axes[3].set_ylabel("condition number")
    axes[3].legend()
else:
    axes[2].text(0.5, 0.5, "solver.outer_history is empty", ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title("Outer Spectral Metrics")
    axes[3].text(0.5, 0.5, "solver.outer_history is empty", ha='center', va='center', transform=axes[3].transAxes)
    axes[3].set_title("Eigenvector Condition Number")

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Training Diagnostics", fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
print("Training diagnostics are plotted in the previous cell.")


In [ ]:
import os
import numpy as np
import tensorflow as tf

export_checkpoint_mode = "best"  # supported: "best", "final", "current"


def restore_solver_checkpoint_for_export(solver, checkpoint_prefix, label):
    # Restore the current solver/model state from a saved checkpoint before exporting outputs.
    if not checkpoint_prefix:
        print(f"No {label} checkpoint path is available; keeping the current in-memory solver state.")
        return False

    checkpoint_dir = os.path.dirname(checkpoint_prefix)
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint is None:
        print(f"No checkpoint found under {checkpoint_dir}; keeping the current in-memory solver state.")
        return False

    K_var = tf.Variable(solver.K, trainable=False, name='K')
    reg_var = tf.Variable(solver.reg, trainable=False, name='reg')
    checkpoint = tf.train.Checkpoint(
        model=solver.model,
        K=K_var,
        eigenvectors=solver.eigenvectors_var,
        eigenvalues=solver.eigenvalues_var,
        reg=reg_var
    )
    checkpoint.restore(latest_checkpoint)

    solver.K = tf.identity(K_var.read_value())
    solver.eigenvectors = solver.eigenvectors_var.numpy().astype(np.complex128, copy=False)
    solver.eigenvalues = solver.eigenvalues_var.numpy().astype(np.complex128, copy=False)
    solver.reg = float(reg_var.numpy())
    solver.eigvec_cond = float(np.linalg.cond(solver.eigenvectors))
    solver.compute_mode()
    solver._sync_training_spectral_state()
    print(f"Restored {label} checkpoint for export: {latest_checkpoint}")
    return True


if export_checkpoint_mode == "best":
    restore_solver_checkpoint_for_export(solver, best_checkpoint_path, "best")
elif export_checkpoint_mode == "final":
    restore_solver_checkpoint_for_export(solver, checkpoint_path, "final")
else:
    print("Keeping the current in-memory solver state for export.")

print("Psi_X and Psi_Y are not materialized in memory.")
print("They will be computed and saved chunk-by-chunk in the final Psi saving cell.")


In [ ]:
export_checkpoint_mode

In [ ]:
chunk_size = 5000
import os
import gc
import h5py
import numpy as np
import scipy.io
from pathlib import Path

dataset_key = field_name if field_name.startswith("/") else f"/{field_name}"

def _get_num_samples_for_export():
    if "DATA" in globals():
        return int(DATA.shape[0])
    with h5py.File(data_full_path, "r") as f:
        return int(f[dataset_key].shape[1])

def _load_observable_chunk(start_idx, end_idx):
    if "DATA" in globals():
        return np.asarray(DATA[start_idx:end_idx, :], dtype=np.float32)
    with h5py.File(data_full_path, "r") as f:
        chunk = np.array(f[dataset_key][:, start_idx:end_idx]).T
    return chunk.astype(np.float32, copy=False)

num_samples = _get_num_samples_for_export()
num_chunks = int(np.ceil(num_samples / float(chunk_size)))

evalues = np.asarray(solver.eigenvalues.T, dtype=np.complex64)
kpm_modes = np.asarray(solver.compute_mode().T, dtype=np.complex64)
loss_array = np.asarray(loss_all, dtype=np.float32)
val_loss_array = np.asarray(val_loss_all, dtype=np.float32)
N_dict = int(np.shape(evalues)[0])
out_base = Path(data_filename).stem


def _as_scalar_or_nan(value):
    # Convert a value to a MATLAB-friendly scalar, or NaN on failure.
    try:
        if value is None:
            return np.nan
        arr = np.asarray(value).squeeze()
        if arr.size == 0:
            return np.nan
        if np.iscomplexobj(arr):
            arr = np.real(arr)
        return float(arr.item())
    except Exception:
        return np.nan


def _history_array(history, key):
    # Extract one outer-history field as a float array.
    return np.asarray(
        [_as_scalar_or_nan(item.get(key, np.nan)) for item in history],
        dtype=np.float32
    )


# Extract outer-loop diagnostics with safe fallbacks.
outer_history = getattr(solver, "outer_history", None)
if outer_history is None:
    outer_history = []

residual_form = str(getattr(solver, "residual_form", ""))
final_eigvec_cond = np.float32(_as_scalar_or_nan(getattr(solver, "eigvec_cond", np.nan)))
best_checkpoint_path = str(getattr(solver, "best_checkpoint_path", "") or "")
final_checkpoint_path = str(getattr(solver, "final_checkpoint_path", "") or "")
num_outer_epochs = int(len(outer_history))

if num_outer_epochs > 0:
    outer_epoch_history = _history_array(outer_history, "outer_epoch")
    outer_train_metric_history = _history_array(outer_history, "train_metric")
    outer_val_metric_history = _history_array(outer_history, "val_metric")
    outer_eigvec_cond_history = _history_array(outer_history, "eigvec_cond")
    outer_lr_history = _history_array(outer_history, "lr")
    outer_reg_history = _history_array(outer_history, "reg")
    outer_inner_train_last_history = _history_array(outer_history, "inner_train_last")
    outer_inner_val_last_history = _history_array(outer_history, "inner_val_last")

    finite_val_mask = np.isfinite(outer_val_metric_history)
    if np.any(finite_val_mask):
        val_candidates = np.where(finite_val_mask, outer_val_metric_history, np.inf)
        best_idx = int(np.argmin(val_candidates))
        best_outer_index = best_idx
        best_outer_epoch = np.float32(outer_epoch_history[best_idx])
        best_val_metric = np.float32(outer_val_metric_history[best_idx])
        best_train_metric_at_best = np.float32(outer_train_metric_history[best_idx])
        best_eigvec_cond_at_best = np.float32(outer_eigvec_cond_history[best_idx])
        best_lr_at_best = np.float32(outer_lr_history[best_idx])
        best_reg_at_best = np.float32(outer_reg_history[best_idx])
    else:
        best_outer_index = np.nan
        best_outer_epoch = np.nan
        best_val_metric = np.nan
        best_train_metric_at_best = np.nan
        best_eigvec_cond_at_best = np.nan
        best_lr_at_best = np.nan
        best_reg_at_best = np.nan
else:
    empty_history = np.array([], dtype=np.float32)
    outer_epoch_history = empty_history
    outer_train_metric_history = empty_history
    outer_val_metric_history = empty_history
    outer_eigvec_cond_history = empty_history
    outer_lr_history = empty_history
    outer_reg_history = empty_history
    outer_inner_train_last_history = empty_history
    outer_inner_val_last_history = empty_history

    best_outer_index = np.nan
    best_outer_epoch = np.nan
    best_val_metric = np.nan
    best_train_metric_at_best = np.nan
    best_eigvec_cond_at_best = np.nan
    best_lr_at_best = np.nan
    best_reg_at_best = np.nan


# Shared outputs saved into one summary file and into every chunk file.
shared_outputs = {
    "evalues": evalues,
    "kpm_modes": kpm_modes,
    "N_dict": N_dict,
    "loss": loss_array,
    "val_loss": val_loss_array,
    "residual_form": residual_form,
    "num_outer_epochs": num_outer_epochs,
    "outer_epoch_history": outer_epoch_history,
    "outer_train_metric_history": outer_train_metric_history,
    "outer_val_metric_history": outer_val_metric_history,
    "outer_eigvec_cond_history": outer_eigvec_cond_history,
    "outer_lr_history": outer_lr_history,
    "outer_reg_history": outer_reg_history,
    "outer_inner_train_last_history": outer_inner_train_last_history,
    "outer_inner_val_last_history": outer_inner_val_last_history,
    "best_outer_index": best_outer_index,
    "best_outer_epoch": best_outer_epoch,
    "best_val_metric": best_val_metric,
    "best_train_metric_at_best": best_train_metric_at_best,
    "best_eigvec_cond_at_best": best_eigvec_cond_at_best,
    "best_lr_at_best": best_lr_at_best,
    "best_reg_at_best": best_reg_at_best,
    "final_eigvec_cond": final_eigvec_cond,
    "best_checkpoint_path": best_checkpoint_path,
    "final_checkpoint_path": final_checkpoint_path,
}

summary_file = os.path.join(
    output_root,
    f"{out_base}_Python_resdmd_Layer_100_Ndict_{N_dict}_summary.mat"
)
scipy.io.savemat(summary_file, {"EDMD_outputs": shared_outputs})
print(f"saved summary: {summary_file}")

for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, num_samples)

    chunk_data = _load_observable_chunk(start_idx, end_idx)
    efuns = np.asarray(solver.eigenfunctions(chunk_data), dtype=np.complex64)

    EDMD_outputs = {
        "EDMD_outputs": {
            "efuns": efuns,
            **shared_outputs,
        }
    }

    out_file = os.path.join(
        output_root,
        f"{out_base}_Python_resdmd_Layer_100_Ndict_{N_dict}_outputs_{i+1}.mat"
    )
    scipy.io.savemat(out_file, EDMD_outputs)
    print(f"saved: {out_file}")

    del chunk_data
    del efuns
    del EDMD_outputs
    gc.collect()


In [ ]:
# chunk_size = 5000
# import os
# import gc
# import h5py
# import numpy as np
# import scipy.io
# from pathlib import Path

# dataset_key = field_name if field_name.startswith("/") else f"/{field_name}"

# def _get_num_samples_for_psi_export():
#     if "DATA" in globals():
#         return int(DATA.shape[0])
#     with h5py.File(data_full_path, "r") as f:
#         return int(f[dataset_key].shape[1])

# def _load_observable_chunk_for_psi(start_idx, end_idx):
#     if "DATA" in globals():
#         return np.asarray(DATA[start_idx:end_idx, :], dtype=np.float32)
#     with h5py.File(data_full_path, "r") as f:
#         chunk = np.array(f[dataset_key][:, start_idx:end_idx]).T
#     return chunk.astype(np.float32, copy=False)

# num_samples = _get_num_samples_for_psi_export()
# num_chunks = int(np.ceil(num_samples / float(chunk_size)))

# evalues = solver.eigenvalues.T
# kpm_modes = solver.compute_mode().T
# N_dict = np.shape(evalues)[0]
# out_base = Path(data_filename).stem

# for i in range(num_chunks):
#     start_idx = i * chunk_size
#     end_idx = min((i + 1) * chunk_size, num_samples)

#     chunk_data = _load_observable_chunk_for_psi(start_idx, end_idx)
#     Psi = solver.dic.call(chunk_data).numpy().T.astype(np.float32, copy=False)
#     Psi_X = Psi[:, 0:-1]
#     Psi_Y = Psi[:, 1:]
#     EDMD_outputs = {
#         'EDMD_outputs': {
#             'Psi_X': Psi_X,
#             'Psi_Y': Psi_Y
#         }
#     }

#     out_file = os.path.join(
#         output_root,
#         f"{out_base}_Python_resdmd_Layer_100_Ndict_{N_dict}_outputs_Psi_{i+1}.mat"
#     )
#     scipy.io.savemat(out_file, EDMD_outputs)
#     print(f"saved: {out_file}")

#     del chunk_data
#     del Psi
#     del Psi_X
#     del Psi_Y
#     del EDMD_outputs
#     gc.collect()
